
# GP Geo CV Tuning

Nested geographic tuning workflow. The expensive cells are restartable: each `(param_id, fold_idx)` fit writes one CSV under `../eval_results/`, and reruns skip completed files.


In [1]:

import json
import os
import subprocess
import sys
import time
from pathlib import Path

import jax.numpy as jnp
import jax.random as jr
import numpy as np
import pandas as pd

CWD = Path.cwd()
if CWD.name == "gridsearch":
    NOTEBOOK_DIR = CWD
elif (CWD / "gridsearch").exists() and CWD.name == "code":
    NOTEBOOK_DIR = CWD / "gridsearch"
elif (CWD / "code" / "gridsearch").exists():
    NOTEBOOK_DIR = CWD / "code" / "gridsearch"
else:
    NOTEBOOK_DIR = CWD
CODE_DIR = NOTEBOOK_DIR.parent

sys.path.append(str(CODE_DIR))

from gp import GaussianProcess
from gp_kernels import make_indoor_outdoor_mean, make_wifi_kernel
from eval_workflow import (
    completed_gp_results,
    load_fold_arrays,
    load_result_index,
    materialize_matching_result,
    summarize_cv_results,
    write_result_row,
)


In [2]:

SPLIT_ROOT = CODE_DIR / "eval_splits" / "geo_outer0"
GRID_PATH = CODE_DIR / "gridsearch" / "gp_geo_stage1_grid.csv"
STAGE1_RESULT_DIR = CODE_DIR / "eval_results" / "gp_stage1"
STAGE2_RESULT_DIR = CODE_DIR / "eval_results" / "gp_stage2"
FINAL_RESULT_PATH = CODE_DIR / "eval_results" / "gp_final_holdout.csv"

STAGE1_FOLDS = 3
STAGE2_FOLDS = 5
STAGE2_TOP_N = 25

N_CHAINS = 2
N_SAMPLES = 111
BURNIN = 10
THIN = 10
CALIBRATION_ITERS = 30
JITTER_FACTOR = 10
PREDICT_METHOD = "sequential"
KEY_SEED = 305


RUN_FITS_IN_SUBPROCESS = True
# Sequential chunking: one subprocess runs this many fits, exits to release GPU memory,
# then the notebook launches the next subprocess. This is not parallel.
FITS_PER_PROCESS = 20
GP_FIT_RUNNER = CODE_DIR / "run_gp_cv_fit.py"
PYTHON_EXECUTABLE = sys.executable


In [3]:

grid = pd.read_csv(GRID_PATH)
grid.shape, grid.head()


((108, 9),
    param_id  ls_xy  ls_z  os_xyz  ls_t  os_t ap_form  ls_ap  os_ap
 0         0  0.025  0.25      20    50     0     add   0.25   10.0
 1         1  0.025  0.25      20    50     0     add   0.25   20.0
 2         2  0.025  0.25      20    50     0     add   0.25   40.0
 3         3  0.025  0.25      20    50     0     add   0.50   10.0
 4         4  0.025  0.25      20    50     0     add   0.50   20.0)

In [4]:

def format_seconds(seconds):
    seconds = int(round(float(seconds)))
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    if hours:
        return f"{hours}h {minutes:02d}m {seconds:02d}s"
    if minutes:
        return f"{minutes}m {seconds:02d}s"
    return f"{seconds}s"


def print_loop_progress(stage_label, completed_count, total_count, loop_start, session_completed):
    elapsed = time.time() - loop_start
    avg = elapsed / max(session_completed, 1)
    remaining = max(total_count - completed_count, 0)
    eta = avg * remaining if session_completed else float("nan")
    eta_text = format_seconds(eta) if session_completed else "unknown"
    print(
        f"{stage_label} progress: {completed_count}/{total_count} complete; "
        f"session elapsed {format_seconds(elapsed)}; "
        f"avg/new fit {format_seconds(avg)}; ETA {eta_text}",
        flush=True,
    )


def count_completed_fits(param_grid, n_folds, result_dir, result_index):
    count = 0
    for _, param_row in param_grid.iterrows():
        for fold_idx in range(n_folds):
            count += materialize_matching_result(result_dir, param_row, fold_idx, result_index) is not None
    return int(count)


def pending_jobs(param_grid, n_folds, result_dir, result_index):
    jobs = []
    for _, param_row in param_grid.iterrows():
        param_id = int(param_row["param_id"])
        for fold_idx in range(n_folds):
            if materialize_matching_result(result_dir, param_row, fold_idx, result_index) is None:
                jobs.append({"param_id": param_id, "fold_idx": int(fold_idx)})
    return jobs


def job_chunks(jobs, chunk_size):
    for start in range(0, len(jobs), int(chunk_size)):
        yield jobs[start:start + int(chunk_size)]


def run_fit_chunk(stage_label, chunk, split_dir, result_dir):
    # subprocess.run is blocking, so chunks run sequentially: at most one GPU process.
    cmd = [
        PYTHON_EXECUTABLE,
        str(GP_FIT_RUNNER),
        "--grid-path", str(GRID_PATH),
        "--split-dir", str(split_dir),
        "--result-dir", str(result_dir),
        "--jobs-json", json.dumps(chunk),
        "--chains", str(N_CHAINS),
        "--samples", str(N_SAMPLES),
        "--burnin", str(BURNIN),
        "--thin", str(THIN),
        "--calibration-iters", str(CALIBRATION_ITERS),
        "--jitter-factor", str(JITTER_FACTOR),
        "--predict-method", PREDICT_METHOD,
        "--key-seed", str(KEY_SEED),
    ]
    env = os.environ.copy()
    env.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
    env.setdefault("XLA_PYTHON_CLIENT_ALLOCATOR", "platform")
    print(
        f"{stage_label}: launching one subprocess for {len(chunk)} fit(s): "
        f"param {chunk[0]['param_id']} fold {chunk[0]['fold_idx']} -> "
        f"param {chunk[-1]['param_id']} fold {chunk[-1]['fold_idx']}",
        flush=True,
    )
    subprocess.run(cmd, check=True, env=env)


def run_stage_loop(stage_label, param_grid, n_folds, split_dir, result_dir):
    total = len(param_grid) * n_folds
    result_index = load_result_index(result_dir)
    completed_count = count_completed_fits(param_grid, n_folds, result_dir, result_index)
    jobs = pending_jobs(param_grid, n_folds, result_dir, result_index)
    loop_start = time.time()
    session_completed = 0
    print(f"Completed {completed_count} / {total} {stage_label.lower()} fits")

    if RUN_FITS_IN_SUBPROCESS:
        for chunk in job_chunks(jobs, FITS_PER_PROCESS):
            run_fit_chunk(stage_label, chunk, split_dir, result_dir)
            session_completed += len(chunk)
            completed_count += len(chunk)
            print_loop_progress(stage_label, completed_count, total, loop_start, session_completed)
        return

    for job in jobs:
        param_row = param_grid.loc[param_grid["param_id"] == job["param_id"]].iloc[0]
        print(f"{stage_label} param {job['param_id']} fold {job['fold_idx']}", flush=True)
        row = fit_evaluate_gp(param_row, split_dir, job["fold_idx"], result_dir)
        session_completed += 1
        completed_count += 1
        print(
            f"{stage_label} fit timing: total={format_seconds(row['total_elapsed'])}, "
            f"gibbs={format_seconds(row['fit_elapsed'])}, "
            f"predict={format_seconds(row['predict_elapsed'])}, mse={row['mse']:.4f}",
            flush=True,
        )
        print_loop_progress(stage_label, completed_count, total, loop_start, session_completed)


def fit_evaluate_gp(param_row, split_dir, fold_idx, result_dir):
    param_id = int(param_row["param_id"])
    completed_row = materialize_matching_result(result_dir, param_row, fold_idx)
    if completed_row is not None:
        return completed_row

    total_start = time.time()
    arrays = load_fold_arrays(split_dir, fold_idx)
    X_train = jnp.asarray(arrays["X_train"])
    y_train = jnp.asarray(arrays["y_train"])
    obs_count_train = jnp.asarray(arrays["obs_count_train"])
    obs_sse_train = jnp.asarray(arrays["obs_sse_train"])
    X_test = jnp.asarray(arrays["X_test"])
    y_test = jnp.asarray(arrays["y_test"])

    m = make_indoor_outdoor_mean(X_train, y_train)
    K = make_wifi_kernel(
        ap_form=param_row["ap_form"],
        ls_xy=param_row["ls_xy"],
        ls_z=param_row["ls_z"],
        os_xyz=param_row["os_xyz"],
        ls_t=param_row["ls_t"],
        os_t=param_row["os_t"],
        ls_ap=param_row["ls_ap"],
        os_ap=param_row["os_ap"],
    )

    gp = GaussianProcess(m, K)
    gp.fit(X_train, y_train, obs_count_train, obs_sse=obs_sse_train)

    start = time.time()
    chain = gp.gibbs(
        key=jr.PRNGKey(KEY_SEED + 1000 * param_id + fold_idx),
        chains=N_CHAINS,
        samples=N_SAMPLES,
        calibration_iters=CALIBRATION_ITERS,
        jitter_factor=JITTER_FACTOR,
    )
    fit_elapsed = time.time() - start

    cov_chains = chain[1][:, BURNIN::THIN, :]
    start = time.time()
    pred_means, pred_vars = gp.predict(X_test, cov_chains, method=PREDICT_METHOD)
    predict_elapsed = time.time() - start

    y_hat = pred_means.mean(axis=0)
    mse = float(jnp.mean((y_test - y_hat) ** 2))

    row = {
        "param_id": param_id,
        "fold_idx": int(fold_idx),
        "n_train": int(X_train.shape[0]),
        "n_test": int(X_test.shape[0]),
        "mse": mse,
        "fit_elapsed": float(fit_elapsed),
        "predict_elapsed": float(predict_elapsed),
        "total_elapsed": float(time.time() - total_start),
        "jitter": float(gp.jitter),
        **{col: param_row[col] for col in ["ap_form", "ls_xy", "ls_z", "os_xyz", "ls_t", "os_t", "ls_ap", "os_ap"]},
    }
    write_result_row(out_path, row)
    return row


In [5]:

# Expensive: stage-1 3-fold GP CV over all configs. Restartable.
stage1_split_dir = SPLIT_ROOT / "inner_geo3"
run_stage_loop("Stage 1", grid, STAGE1_FOLDS, stage1_split_dir, STAGE1_RESULT_DIR)


Completed 0 / 324 stage 1 fits
Stage 1: launching one subprocess for 20 fit(s): param 0 fold 0 -> param 6 fold 1
Worker starting 20 fit(s)
Worker fit 1/20: param 0 fold 0
Wrote /home/jamie/storage-1/github-repos/where-my-wifi-kevin/code/eval_results/gp_stage1/param_0000_fold_0.csv mse=107.3191 total=10.5s
Worker fit 2/20: param 0 fold 1
Wrote /home/jamie/storage-1/github-repos/where-my-wifi-kevin/code/eval_results/gp_stage1/param_0000_fold_1.csv mse=84.6518 total=8.5s
Worker fit 3/20: param 0 fold 2
Wrote /home/jamie/storage-1/github-repos/where-my-wifi-kevin/code/eval_results/gp_stage1/param_0000_fold_2.csv mse=78.4593 total=8.9s
Worker fit 4/20: param 1 fold 0
Wrote /home/jamie/storage-1/github-repos/where-my-wifi-kevin/code/eval_results/gp_stage1/param_0001_fold_0.csv mse=109.5895 total=5.7s
Worker fit 5/20: param 1 fold 1
Wrote /home/jamie/storage-1/github-repos/where-my-wifi-kevin/code/eval_results/gp_stage1/param_0001_fold_1.csv mse=nan total=5.2s
Worker fit 6/20: param 1 fold 2


In [6]:

stage1_results = completed_gp_results(STAGE1_RESULT_DIR, grid)
stage1_summary = summarize_cv_results(stage1_results)
stage1_summary = stage1_summary.merge(grid, on="param_id", how="left")
stage1_complete = stage1_summary[stage1_summary["n_folds"] == STAGE1_FOLDS].copy()
stage1_summary.head(25)


,param_id,mean_mse,std_mse,n_folds,mean_fit_elapsed,mean_predict_elapsed,mean_total_elapsed,ls_xy,ls_z,os_xyz,ls_t,os_t,ap_form,ls_ap,os_ap
0,36,88.909093,16.461432,3,4.365819,0.679917,5.152366,0.035,0.25,20,50,0,add,0.25,10.0
1,54,89.042076,16.407107,3,4.944210,0.991316,6.422425,0.035,0.50,20,50,0,add,0.25,10.0
2,72,89.534884,16.214233,3,4.385564,0.674100,5.164043,0.040,0.25,20,50,0,add,0.25,10.0
3,90,89.571940,16.210778,3,4.383437,0.679755,5.169148,0.040,0.50,20,50,0,add,0.25,10.0
4,39,89.585419,16.993128,3,4.389339,0.688039,5.184214,0.035,0.25,20,50,0,add,0.50,10.0
5,42,89.653496,17.591657,3,4.438399,0.719214,5.264047,0.035,0.25,25,50,0,add,0.25,10.0
6,57,89.692713,16.994261,3,4.401121,0.661619,5.167560,0.035,0.50,20,50,0,add,0.50,10.0
7,60,89.833837,17.514991,3,6.299276,1.680989,9.366570,0.035,0.50,25,50,0,add,0.25,10.0
8,75,89.984528,16.737646,3,4.382577,0.685301,5.173503,0.040,0.25,20,50,0,add,0.50,10.0
9,93,90.095673,16.757307,3,5.788617,1.362619,8.158262,0.040,0.50,20,50,0,add,0.50,10.0


In [7]:

if len(stage1_complete) < STAGE2_TOP_N:
    raise RuntimeError(
        f"Need {STAGE2_TOP_N} complete stage-1 configs before stage 2; "
        f"found {len(stage1_complete)}."
    )
stage2_param_ids = stage1_complete.head(STAGE2_TOP_N)["param_id"].astype(int).to_list()
stage2_grid = grid[grid["param_id"].isin(stage2_param_ids)].copy()
stage2_grid


,param_id,ls_xy,ls_z,os_xyz,ls_t,os_t,ap_form,ls_ap,os_ap
0,0,0.025,0.25,20,50,0,add,0.25,10.0
3,3,0.025,0.25,20,50,0,add,0.50,10.0
6,6,0.025,0.25,25,50,0,add,0.25,10.0
9,9,0.025,0.25,25,50,0,add,0.50,10.0
18,18,0.025,0.50,20,50,0,add,0.25,10.0
21,21,0.025,0.50,20,50,0,add,0.50,10.0
24,24,0.025,0.50,25,50,0,add,0.25,10.0
27,27,0.025,0.50,25,50,0,add,0.50,10.0
36,36,0.035,0.25,20,50,0,add,0.25,10.0
37,37,0.035,0.25,20,50,0,add,0.25,20.0


In [8]:

# Expensive: stage-2 5-fold GP CV over the top 25 stage-1 configs. Restartable.
stage2_split_dir = SPLIT_ROOT / "inner_geo5"
run_stage_loop("Stage 2", stage2_grid, STAGE2_FOLDS, stage2_split_dir, STAGE2_RESULT_DIR)


Completed 0 / 125 stage 2 fits
Stage 2: launching one subprocess for 20 fit(s): param 0 fold 0 -> param 9 fold 4
Worker starting 20 fit(s)
Worker fit 1/20: param 0 fold 0
Wrote /home/jamie/storage-1/github-repos/where-my-wifi-kevin/code/eval_results/gp_stage2/param_0000_fold_0.csv mse=84.2933 total=11.4s
Worker fit 2/20: param 0 fold 1
Wrote /home/jamie/storage-1/github-repos/where-my-wifi-kevin/code/eval_results/gp_stage2/param_0000_fold_1.csv mse=133.1027 total=9.7s
Worker fit 3/20: param 0 fold 2
Wrote /home/jamie/storage-1/github-repos/where-my-wifi-kevin/code/eval_results/gp_stage2/param_0000_fold_2.csv mse=79.0707 total=9.2s
Worker fit 4/20: param 0 fold 3
Wrote /home/jamie/storage-1/github-repos/where-my-wifi-kevin/code/eval_results/gp_stage2/param_0000_fold_3.csv mse=71.6227 total=10.0s
Worker fit 5/20: param 0 fold 4
Wrote /home/jamie/storage-1/github-repos/where-my-wifi-kevin/code/eval_results/gp_stage2/param_0000_fold_4.csv mse=74.1598 total=10.2s
Worker fit 6/20: param 3 fo

In [9]:

stage2_results = completed_gp_results(STAGE2_RESULT_DIR, grid)
stage2_summary = summarize_cv_results(stage2_results)
stage2_summary = stage2_summary.merge(grid, on="param_id", how="left")
stage2_complete = stage2_summary[stage2_summary["n_folds"] == STAGE2_FOLDS].copy()
stage2_summary.head(25)


,param_id,mean_mse,std_mse,n_folds,mean_fit_elapsed,mean_predict_elapsed,mean_total_elapsed,ls_xy,ls_z,os_xyz,ls_t,os_t,ap_form,ls_ap,os_ap
0,72,87.246924,17.339720,5,4.936885,0.834561,5.871410,0.040,0.25,20,50,0,add,0.25,10.0
1,36,87.281126,19.731619,5,6.980022,1.772672,10.045219,0.035,0.25,20,50,0,add,0.25,10.0
2,90,87.288657,17.131644,5,4.903731,0.795862,5.801117,0.040,0.50,20,50,0,add,0.25,10.0
3,54,87.299750,19.531152,5,4.973888,0.810807,5.888221,0.035,0.50,20,50,0,add,0.25,10.0
4,75,87.462367,18.099614,5,4.939185,0.828987,5.868620,0.040,0.25,20,50,0,add,0.50,10.0
5,93,87.523151,17.887483,5,4.921200,0.806215,5.830599,0.040,0.50,20,50,0,add,0.50,10.0
6,39,87.643794,20.690397,5,4.988151,0.830359,5.918282,0.035,0.25,20,50,0,add,0.50,10.0
7,57,87.745233,20.562092,5,4.925032,0.814084,5.838898,0.035,0.50,20,50,0,add,0.50,10.0
8,0,88.449838,25.428730,5,6.988316,1.808703,10.096492,0.025,0.25,20,50,0,add,0.25,10.0
9,18,88.564130,25.386417,5,6.968622,1.747736,10.037728,0.025,0.50,20,50,0,add,0.25,10.0


In [10]:

if stage2_complete.empty:
    raise RuntimeError("Need at least one complete stage-2 config before final holdout evaluation.")
best_params = stage2_complete.iloc[0]
best_params


param_id                       72
mean_mse                87.246924
std_mse                  17.33972
n_folds                         5
mean_fit_elapsed         4.936885
mean_predict_elapsed     0.834561
mean_total_elapsed        5.87141
ls_xy                        0.04
ls_z                         0.25
os_xyz                         20
ls_t                           50
os_t                            0
ap_form                       add
ls_ap                        0.25
os_ap                        10.0
Name: 0, dtype: object

In [11]:

def load_outer(prefix):
    return {
        "X": jnp.asarray(np.load(SPLIT_ROOT / f"{prefix}_X.npy")),
        "y": jnp.asarray(np.load(SPLIT_ROOT / f"{prefix}_y.npy")),
        "obs_count": jnp.asarray(np.load(SPLIT_ROOT / f"{prefix}_obs_count.npy")),
        "obs_sse": jnp.asarray(np.load(SPLIT_ROOT / f"{prefix}_obs_sse.npy")),
    }

outer_train = load_outer("outer_train")
outer_holdout = load_outer("outer_holdout")

m = make_indoor_outdoor_mean(outer_train["X"], outer_train["y"])
K = make_wifi_kernel(
    ap_form=best_params["ap_form"],
    ls_xy=best_params["ls_xy"],
    ls_z=best_params["ls_z"],
    os_xyz=best_params["os_xyz"],
    ls_t=best_params["ls_t"],
    os_t=best_params["os_t"],
    ls_ap=best_params["ls_ap"],
    os_ap=best_params["os_ap"],
)

gp = GaussianProcess(m, K)
gp.fit(outer_train["X"], outer_train["y"], outer_train["obs_count"], obs_sse=outer_train["obs_sse"])

start = time.time()
chain = gp.gibbs(
    key=jr.PRNGKey(KEY_SEED + 999_999),
    chains=N_CHAINS,
    samples=N_SAMPLES,
    calibration_iters=CALIBRATION_ITERS,
    jitter_factor=JITTER_FACTOR,
)
fit_elapsed = time.time() - start

cov_chains = chain[1][:, BURNIN::THIN, :]
start = time.time()
pred_means, pred_vars = gp.predict(outer_holdout["X"], cov_chains, method=PREDICT_METHOD)
predict_elapsed = time.time() - start

y_hat = pred_means.mean(axis=0)
final_mse = float(jnp.mean((outer_holdout["y"] - y_hat) ** 2))
final_row = {
    "model": "gp",
    "param_id": int(best_params["param_id"]),
    "mse": final_mse,
    "n_train": int(outer_train["X"].shape[0]),
    "n_test": int(outer_holdout["X"].shape[0]),
    "fit_elapsed": float(fit_elapsed),
    "predict_elapsed": float(predict_elapsed),
    "jitter": float(gp.jitter),
    **{col: best_params[col] for col in ["ap_form", "ls_xy", "ls_z", "os_xyz", "ls_t", "os_t", "ls_ap", "os_ap"]},
}
write_result_row(FINAL_RESULT_PATH, final_row)
final_row


{'model': 'gp',
 'param_id': 652,
 'mse': 73.17362976074219,
 'n_train': 2064,
 'n_test': 553,
 'fit_elapsed': 7.942957401275635,
 'predict_elapsed': 1.7508141994476318,
 'jitter': 0.0008801842923276126,
 'ap_form': 'mult',
 'ls_xy': np.float64(0.05),
 'ls_z': np.float64(0.25),
 'os_xyz': np.int64(20),
 'ls_t': np.int64(50),
 'os_t': np.int64(0),
 'ls_ap': np.float64(0.5),
 'os_ap': np.float64(0.0)}